# Parcial – Simulador de Ads (plantilla)

**Nombre:** ______________________  **Fecha:** __________

Lea primero `ENUNCIADO.md`. Cada `# TODO` es trabajo suyo. Al final, el simulador
(`streamlit run app.py`) debe mostrar 🟢 en los tres modelos.

| Parte | Modelo | Archivo que debe guardar en `modelos/entrenados/` |
|---|---|---|
| A | CNN: tipo de imagen (personas / producto / descuento) | `cnn_imagenes.keras` + `cnn_config.json` |
| B | LSTM: tipo de mensaje (promocion / caracteristicas / descuento) + regresión logística (línea base) | `mensaje_lstm.keras` + `mensaje_lstm_config.json`, `mensaje_lr.joblib` |
| C | Random Forest: ¿la tasa de interacción superará el promedio? | `ads_random_forest.joblib` |

In [ ]:
import glob, json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay)

SEED = 42
BASE = Path("..")
DATA = BASE / "data"
SALIDA = BASE / "modelos" / "entrenados"
SALIDA.mkdir(parents=True, exist_ok=True)

---
## Parte 0 – Las métricas (lea la sección 2 del enunciado)
`tasa_interaccion = (likes + comentarios) / seguidores` es nuestro proxy del CTR.
`poner_ad = 1` si la tasa del post supera el **promedio de su cuenta**.

In [ ]:
pub = pd.read_csv(DATA / "ads" / "ads_publicados.csv", parse_dates=["fecha"])
# TODO 0.1: distribución de likes, comentarios y tasa_interaccion por cuenta (histogramas en escala log)
# TODO 0.2: ¿por qué comparamos contra el promedio de CADA cuenta y no contra el promedio global?
# TODO 0.3: ¿qué proporción de posts tiene poner_ad = 1? ¿por qué no es 50%? (pista: media vs mediana)
prod = pd.read_csv(DATA / "ads" / "productos_alkosto.csv")
# TODO 0.4: ventas_proxy (opiniones) por categoría y marca en Alkosto: ¿qué marcas dominan cada categoría?

---
## Parte A – CNN: tipo de imagen (10%)
Imágenes reales de anuncios (Meta Ad Library) y de publicaciones de Instagram. La carpeta es la etiqueta.
Referencias: `src/notebook/CNN.ipynb`, `src/notebook_class/supervisado/supervisado_neural_nets.ipynb`.

In [ ]:
CLASES = ["personas", "producto", "descuento"]
IMG_SIZE = (64, 64)   # TODO: experimente (las imágenes originales son 224x224)
archivos = glob.glob(str(DATA / "imagenes" / "train" / "*" / "*.jpg"))
etiquetas = [Path(f).parent.name for f in archivos]
pd.Series(etiquetas).value_counts()

In [ ]:
X_img = np.array([np.array(Image.open(f).convert("RGB").resize(IMG_SIZE)) for f in archivos], dtype="float32") / 255.0
y_img = np.array([CLASES.index(e) for e in etiquetas])
X_tr, X_val, y_tr, y_val = train_test_split(X_img, y_img, test_size=0.2, random_state=SEED, stratify=y_img)
# TODO A.1 Líneas base: clase mayoritaria y regresión logística sobre píxeles (accuracy, F1 macro, matriz de confusión)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
# Data augmentation DENTRO del modelo (solo actúa al entrenar) + class_weight
# aug = models.Sequential([layers.RandomRotation(0.05), layers.RandomZoom(0.1), layers.RandomContrast(0.2),
#                          layers.RandomBrightness(0.2, value_range=(0, 1)), layers.GaussianNoise(0.05)])
# pesos = {i: len(y_tr) / (3 * (y_tr == i).sum()) for i in range(3)}
# TODO A.2 CNN (aug -> Conv2D -> MaxPooling -> ... -> Dense(3, softmax)); curvas loss/accuracy; matriz de confusión
# TODO A.2b Compare las curvas de validación con y sin augmentation.
# TODO A.3 (opcional) transfer learning con MobileNetV2
# cnn.save(SALIDA / "cnn_imagenes.keras")
# json.dump({"img_size": list(IMG_SIZE), "color": "rgb", "clases": CLASES, "normalizar": True},
#           open(SALIDA / "cnn_config.json", "w"))

In [ ]:
# El test es OCULTO: el profesor corre su cnn_imagenes.keras. Verifique su flujo con la muestra de 10:
muestra = pd.read_csv(DATA / "imagenes" / "muestra_test_etiquetas.csv")
# TODO A.4 prediga DATA/"imagenes"/"muestra_test"/<archivo> y compare con muestra["tipo_imagen"]
# TODO A.5 reporte el F1 macro de VALIDACIÓN de su CNN final (se compara con el del test oculto)

---
## Parte B – Tipo de mensaje: LSTM y regresión logística (10%)
Textos reales de anuncios y publicaciones. Clases: `promocion`, `caracteristicas`, `descuento`.
Referencias: `src/notebook_class/supervisado/supervisado_neural_nets.ipynb` (Embedding + LSTM),
`src/notebook_class/supervisado/supervisado_regresion_logistic.ipynb`.

In [ ]:
txt = pd.read_csv(DATA / "textos" / "mensajes_train.csv")
muestra_txt = pd.read_csv(DATA / "textos" / "mensajes_muestra_test.csv")   # 10 textos del test oculto
txt["tipo_mensaje"].value_counts(), pd.crosstab(txt["cuenta"], txt["tipo_mensaje"])

In [ ]:
# Validación DENTRO de train. Split por MARCA: ninguna marca de validación aparece en train (las marcas repiten plantillas de texto)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
i_tr, i_te = next(gss.split(txt, groups=txt["cuenta"]))
# TODO B.0 Compare también con un split aleatorio estratificado. ¿Cuál es más honesto? ¿Por qué?

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
# TODO B.1 Línea base: Pipeline([("vectorizer", TfidfVectorizer(...)), ("clf", LogisticRegression(...))])
#          matriz de confusión + top 10 palabras por clase (coeficientes)
# TODO B.1b Explicabilidad: ¿esos términos tienen sentido para decidir el tipo de mensaje?
# joblib.dump(pipe_lr, SALIDA / "mensaje_lr.joblib")

In [ ]:
CLASES_MSG = ["promocion", "caracteristicas", "descuento"]
# TODO B.2 LSTM: el modelo recibe TEXTO CRUDO
# vectorizador = layers.TextVectorization(max_tokens=..., output_sequence_length=...)
# vectorizador.adapt(textos_train)
# entrada = tf.keras.Input(shape=(1,), dtype=tf.string)
# x = vectorizador(entrada); x = layers.Embedding(...)(x); x = layers.LSTM(...)(x); x = layers.Dropout(0.2)(x)
# salida = layers.Dense(3, activation="softmax")(x)
# TODO: curvas de entrenamiento, matriz de confusión, comparación con la regresión logística
# lstm.save(SALIDA / "mensaje_lstm.keras")
# json.dump({"clases": CLASES_MSG, "normalizar_texto": False}, open(SALIDA / "mensaje_lstm_config.json", "w"))
# El test es OCULTO: el profesor corre sus modelos guardados. Prediga muestra_txt con ambos modelos y
# reporte el F1 macro de VALIDACIÓN del modelo final.

---
## Parte C – ¿Publicamos? Random Forest (14%)
Use **las mismas variables que arma el simulador** (`modelos/variables.py`).

In [ ]:
import sys; sys.path.insert(0, str(BASE))
from modelos.variables import CATEGORICAS, NUMERICAS, FEATURES
print(FEATURES)

In [ ]:
# TODO C.0 Exploración: tasa de éxito por tipo_imagen x tipo_mensaje, por categoria_producto, por hora y día,
#          efecto de es_concurso. Contraste con las hipótesis H1-H5 del enunciado.

In [ ]:
# Split: los posts son una serie de tiempo; compare split aleatorio estratificado vs. split temporal
# (entrenar con los más antiguos y probar con los más recientes).
X_tr, X_te, y_tr, y_te = train_test_split(pub[FEATURES], pub["poner_ad"], test_size=0.25,
                                          random_state=SEED, stratify=pub["poner_ad"])

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
# pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
#                          ("num", SimpleImputer(strategy="median"), NUMERICAS)])
# TODO C.1 Línea base: regresión logística (agregue StandardScaler)
# TODO C.2 Random Forest con GridSearchCV SOLO sobre train (class_weight="balanced")
# TODO C.3 Test: matriz de confusión, accuracy, F1, AUC, ROC; feature importance agrupada
# joblib.dump(pipe_rf, SALIDA / "ads_random_forest.joblib")

In [ ]:
nuevos = pd.read_csv(DATA / "ads" / "ads_nuevos.csv")
# Entregable: entrega/predicciones_ads_nuevos.csv con columnas post_id,p_exito,poner_ad

---
## Parte D – Simulador
`cd src/bigdata_parcial && streamlit run app.py` → verifique 🟢 en los tres modelos y tome capturas.